
# 13 - OLS geographic effects

Central OLS geographic diagnostics notebook.

Question:

> How much does geography matter after controlling for the core individual, labor, education, household, and housing covariates?

Experiments compared:

1. `ols_core`
2. `ols_core_region_fe`
3. `ols_core_aglo_fe`
4. `ols_core_aglo_plus_time_fe`

Main outputs:

- validation/test metric changes vs `ols_core`
- group mean residuals by `Region` and `AGLOMERADO`
- geographic residual variance decomposition
- agglomerate rankings and top contributors
- comparison of agglomerate effects across FE settings


## 00. Setup and run discovery

In [ ]:

from pathlib import Path
import json
import yaml

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/home/matias/repos/income-modeling-eph"),
]

ROOT = next(
    (p for p in ROOT_CANDIDATES if (p / "reports" / "runs").exists()),
    Path("/home/matias/repos/income-modeling-eph"),
)

RUNS_DIR = ROOT / "reports" / "runs"
DATASET_PATH = ROOT / "data" / "processed" / "modeling_dataset.parquet"

OUTPUT_DIR = ROOT / "reports" / "notebook_outputs" / "ols_geographic_effects"
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR = OUTPUT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 160)

print("ROOT:", ROOT)
print("RUNS_DIR:", RUNS_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


def read_json_if_exists(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text())


def read_yaml_if_exists(path: Path):
    if not path.exists():
        return None
    return yaml.safe_load(path.read_text())


def read_csv_if_exists(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.exists() else pd.DataFrame()


def read_parquet_if_exists(path: Path) -> pd.DataFrame:
    return pd.read_parquet(path) if path.exists() else pd.DataFrame()


RUN_PATTERNS = {
    "ols_core": "ols_core_*",
    "ols_core_region_fe": "ols_core_region_fe_*",
    "ols_core_aglo_fe": "ols_core_aglo_fe_*",
    "ols_core_aglo_plus_time_fe": "ols_core_aglo_plus_time_fe_*",
}

ORDER = [
    "ols_core",
    "ols_core_region_fe",
    "ols_core_aglo_fe",
    "ols_core_aglo_plus_time_fe",
]

DISPLAY_LABELS = {
    "ols_core": "Core OLS",
    "ols_core_region_fe": "+ Region FE",
    "ols_core_aglo_fe": "+ Aglomerado FE",
    "ols_core_aglo_plus_time_fe": "+ Aglomerado + time FE",
}


def discover_runs() -> dict[str, Path | None]:
    runs = {}
    for experiment, pattern in RUN_PATTERNS.items():
        candidates = sorted(RUNS_DIR.glob(pattern), key=lambda p: p.name)
        if not candidates:
            runs[experiment] = None
            continue

        # Use config experiment id to avoid broad-pattern collisions.
        exact = []
        for candidate in candidates:
            config = read_yaml_if_exists(candidate / "config_used.yaml") or {}
            experiment_id = ((config.get("experiment") or {}).get("id"))
            if experiment_id == experiment:
                exact.append(candidate)

        runs[experiment] = exact[-1] if exact else candidates[-1]
    return runs


RUNS = discover_runs()

run_table = pd.DataFrame(
    [
        {"experiment": exp, "run_dir": str(path) if path else None, "found": path is not None}
        for exp, path in RUNS.items()
    ]
)

run_table


## 01. Load backend artifacts

In [ ]:

def load_model_comparison(run_dir: Path, experiment: str) -> pd.DataFrame:
    df = read_csv_if_exists(run_dir / "metrics" / "model_comparison.csv")
    if df.empty:
        return df
    df["experiment"] = experiment
    df["run_dir"] = str(run_dir)
    return df


def load_predictions(run_dir: Path, experiment: str, split: str) -> pd.DataFrame:
    path = run_dir / "predictions" / f"{split}_predictions.parquet"
    df = read_parquet_if_exists(path)
    if df.empty:
        return df

    df["experiment"] = experiment
    df["run_dir"] = str(run_dir)
    df["split"] = split

    if "residual" not in df.columns:
        df["residual"] = df["y_true"] - df["y_pred"]
    if "abs_error" not in df.columns:
        df["abs_error"] = df["residual"].abs()
    if "squared_error" not in df.columns:
        df["squared_error"] = df["residual"] ** 2

    return df


def load_run_metadata(run_dir: Path, experiment: str) -> dict:
    feature_columns = read_json_if_exists(run_dir / "feature_columns.json") or []
    dataset_card = read_json_if_exists(run_dir / "dataset_card.json") or {}
    config_used = read_yaml_if_exists(run_dir / "config_used.yaml") or {}
    feature_view = config_used.get("feature_view") or {}
    fixed_effects = (config_used.get("model_design") or {}).get("fixed_effects") or []

    return {
        "experiment": experiment,
        "run_dir": str(run_dir),
        "n_feature_columns": len(feature_columns),
        "feature_columns": feature_columns,
        "feature_view_name": feature_view.get("name"),
        "include_blocks": feature_view.get("include_blocks"),
        "fixed_effects_config": fixed_effects,
        "dataset_card": dataset_card,
        "config_used": config_used,
    }


model_comparison_parts = []
prediction_parts = []
metadata_rows = []

for experiment, run_dir in RUNS.items():
    if run_dir is None:
        continue
    model_comparison_parts.append(load_model_comparison(run_dir, experiment))
    for split in ["validation", "test"]:
        prediction_parts.append(load_predictions(run_dir, experiment, split))
    metadata_rows.append(load_run_metadata(run_dir, experiment))

model_comparison = (
    pd.concat([d for d in model_comparison_parts if not d.empty], ignore_index=True)
    if any(not d.empty for d in model_comparison_parts)
    else pd.DataFrame()
)

predictions = (
    pd.concat([d for d in prediction_parts if not d.empty], ignore_index=True)
    if any(not d.empty for d in prediction_parts)
    else pd.DataFrame()
)

run_metadata = pd.DataFrame(metadata_rows)

print("model_comparison:", model_comparison.shape)
print("predictions:", predictions.shape)
print("run_metadata:", run_metadata.shape)

run_metadata[["experiment", "feature_view_name", "include_blocks", "fixed_effects_config", "n_feature_columns"]]


## 02. Enrich predictions with geography and time

In [ ]:

if predictions.empty:
    raise RuntimeError(
        "No predictions were loaded. Run the OLS geographic experiments first or check RUN_PATTERNS."
    )

geo_cols = ["row_id", "Region", "AGLOMERADO", "ANO4", "TRIMESTRE"]
if DATASET_PATH.exists():
    dataset_geo = pd.read_parquet(DATASET_PATH, columns=geo_cols)
else:
    raise FileNotFoundError(f"Processed dataset not found: {DATASET_PATH}")

pred_geo = predictions.merge(dataset_geo, on="row_id", how="left", validate="many_to_one")

for col in ["Region", "AGLOMERADO", "ANO4", "TRIMESTRE"]:
    pred_geo[col] = pred_geo[col].astype("string")

pred_geo.head()


## 03. Aggregate metrics vs `ols_core`

In [ ]:

def safe_r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    if len(y_true) == 0:
        return np.nan
    denom = np.sum((y_true - y_true.mean()) ** 2)
    if denom == 0:
        return np.nan
    return 1 - np.sum((y_true - y_pred) ** 2) / denom


def aggregate_metrics(g):
    y = g["y_true"].astype(float)
    yhat = g["y_pred"].astype(float)
    residual = y - yhat
    return pd.Series({
        "n": len(g),
        "r2": safe_r2(y, yhat),
        "mae": residual.abs().mean(),
        "rmse": np.sqrt((residual ** 2).mean()),
        "mean_error": (yhat - y).mean(),
        "sd_y_true": y.std(),
        "sd_y_pred": yhat.std(),
        "compression_ratio": yhat.std() / y.std() if y.std() else np.nan,
    })


metrics_by_split = (
    pred_geo
    .groupby(["experiment", "split"], as_index=False)
    .apply(aggregate_metrics, include_groups=False)
    .reset_index(drop=True)
)

metrics_by_split["experiment_order"] = metrics_by_split["experiment"].map({exp: i for i, exp in enumerate(ORDER)})
metrics_by_split["label"] = metrics_by_split["experiment"].map(DISPLAY_LABELS)

validation_summary = (
    metrics_by_split
    .query("split == 'validation'")
    .sort_values("experiment_order")
    .reset_index(drop=True)
)

core = validation_summary.query("experiment == 'ols_core'")
if not core.empty:
    core = core.iloc[0]
    for col in ["r2", "mae", "rmse", "compression_ratio"]:
        validation_summary[f"delta_{col}_vs_core"] = validation_summary[col] - core[col]

metrics_by_split.to_csv(TABLE_DIR / "T1_ols_geo_metrics_by_split.csv", index=False)
validation_summary.to_csv(TABLE_DIR / "T2_ols_geo_validation_vs_core.csv", index=False)
run_metadata.to_csv(TABLE_DIR / "T3_ols_geo_run_metadata.csv", index=False)

validation_summary[[
    "experiment", "label", "n", "r2", "delta_r2_vs_core",
    "mae", "delta_mae_vs_core", "rmse", "delta_rmse_vs_core",
    "compression_ratio", "delta_compression_ratio_vs_core"
]].round(5)


## 04. Figure 1 — validation ΔR² vs core

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(validation_summary["label"], validation_summary["delta_r2_vs_core"])
ax.axhline(0, linewidth=1)
ax.set_title("Geographic FE contribution: validation ΔR² vs core OLS")
ax.set_xlabel("Specification")
ax.set_ylabel("ΔR² vs ols_core")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "F1_ols_geo_delta_r2_vs_core.png", dpi=160)
plt.show()



## 05. Group residual summaries

For a model without group FE, group mean residuals approximate:

\[
\bar{u}_g = \overline{y - \hat{y}}_g
\]

That is, observed group mean minus expected group mean according to non-geographic controls.

For FE models, residual group means after fitting may be closer to zero for included FE groups. We therefore compare group residual structure across settings rather than treating all rows as identical FE estimates.


In [ ]:

def group_residual_summary(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    return (
        df
        .dropna(subset=[group_col])
        .groupby(["experiment", "split", group_col], as_index=False)
        .agg(
            n=("row_id", "size"),
            mean_y_true=("y_true", "mean"),
            mean_y_pred=("y_pred", "mean"),
            mean_residual=("residual", "mean"),
            sd_residual=("residual", "std"),
            mae=("abs_error", "mean"),
            rmse=("squared_error", lambda s: np.sqrt(np.mean(s))),
        )
        .rename(columns={group_col: "group_value"})
        .assign(group_type=group_col)
    )

region_residual_summary = group_residual_summary(pred_geo, "Region")
aglo_residual_summary = group_residual_summary(pred_geo, "AGLOMERADO")

group_residuals = pd.concat([region_residual_summary, aglo_residual_summary], ignore_index=True)
group_residuals["abs_mean_residual"] = group_residuals["mean_residual"].abs()

group_residuals.to_csv(TABLE_DIR / "T4_ols_geo_group_residuals.csv", index=False)

group_residuals.query("split == 'validation' and group_type == 'AGLOMERADO'").head()


## 06. Geographic variance decomposition

In [ ]:

def weighted_var(x, w=None):
    x = np.asarray(x, dtype=float)
    if w is None:
        mask = np.isfinite(x)
        return float(np.nanvar(x[mask])) if mask.any() else np.nan
    w = np.asarray(w, dtype=float)
    mask = np.isfinite(x) & np.isfinite(w)
    if mask.sum() == 0 or w[mask].sum() == 0:
        return np.nan
    mu = np.average(x[mask], weights=w[mask])
    return float(np.average((x[mask] - mu) ** 2, weights=w[mask]))


rows = []
for (experiment, split, group_type), df_g in group_residuals.groupby(["experiment", "split", "group_type"]):
    df_obs = pred_geo.query("experiment == @experiment and split == @split").copy()
    group_col = group_type
    if group_col not in df_obs.columns:
        continue

    effect_map = df_g.set_index("group_value")["mean_residual"]
    df_obs["group_mean_residual"] = df_obs[group_col].map(effect_map)

    var_y = weighted_var(df_obs["y_true"])
    var_pred = weighted_var(df_obs["y_pred"])
    var_resid = weighted_var(df_obs["residual"])
    var_group = weighted_var(df_obs["group_mean_residual"])

    rows.append({
        "experiment": experiment,
        "split": split,
        "group_type": group_type,
        "var_y": var_y,
        "var_pred": var_pred,
        "var_residual": var_resid,
        "var_group_mean_residual": var_group,
        "sd_y": np.sqrt(var_y),
        "sd_residual": np.sqrt(var_resid),
        "sd_group_mean_residual": np.sqrt(var_group),
        "share_var_y": var_group / var_y if var_y else np.nan,
        "share_var_residual": var_group / var_resid if var_resid else np.nan,
        "share_var_y_pct": 100 * var_group / var_y if var_y else np.nan,
        "share_var_residual_pct": 100 * var_group / var_resid if var_resid else np.nan,
    })

geo_variance_decomposition = pd.DataFrame(rows)
geo_variance_decomposition["experiment_order"] = geo_variance_decomposition["experiment"].map({exp: i for i, exp in enumerate(ORDER)})
geo_variance_decomposition["label"] = geo_variance_decomposition["experiment"].map(DISPLAY_LABELS)

geo_variance_decomposition.to_csv(TABLE_DIR / "T5_ols_geo_variance_decomposition.csv", index=False)

geo_variance_decomposition.query("split == 'validation'").sort_values(
    ["group_type", "share_var_y_pct"], ascending=[True, False]
).round(5)


## 07. Figure 2 — geographic residual variance share

In [ ]:

plot_df = (
    geo_variance_decomposition
    .query("split == 'validation'")
    .copy()
)

fig, ax = plt.subplots(figsize=(9, 5))
for group_type, tmp in plot_df.groupby("group_type"):
    tmp = tmp.sort_values("experiment_order")
    ax.plot(tmp["label"], tmp["share_var_y_pct"], marker="o", label=group_type)

ax.set_title("Geographic residual structure: share of total y variance")
ax.set_xlabel("Specification")
ax.set_ylabel("Var(group mean residual) / Var(y) (%)")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "F2_ols_geo_variance_share_region_vs_aglo.png", dpi=160)
plt.show()


## 08. Figure 3 — agglomerate residual ranking

In [ ]:

SPLIT = "validation"
REFERENCE_EXPERIMENT = "ols_core"
MIN_N = 50

df_aglo = (
    group_residuals
    .query("split == @SPLIT and group_type == 'AGLOMERADO' and n >= @MIN_N")
    .copy()
)

ref_order = (
    df_aglo
    .query("experiment == @REFERENCE_EXPERIMENT")
    .sort_values("mean_residual")
    ["group_value"]
    .tolist()
)

if not ref_order:
    ref_order = (
        df_aglo
        .sort_values("mean_residual")
        ["group_value"]
        .drop_duplicates()
        .tolist()
    )

df_aglo["group_order"] = pd.Categorical(df_aglo["group_value"], categories=ref_order, ordered=True)
df_aglo = df_aglo.sort_values(["group_order", "experiment"])

plot_df = df_aglo.query("experiment in ['ols_core', 'ols_core_aglo_fe', 'ols_core_aglo_plus_time_fe']").copy()

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(ref_order))

for experiment in ["ols_core", "ols_core_aglo_fe", "ols_core_aglo_plus_time_fe"]:
    tmp = plot_df.query("experiment == @experiment").sort_values("group_order")
    if tmp.empty:
        continue
    ax.plot(x, tmp["mean_residual"], marker=".", linewidth=1, label=DISPLAY_LABELS.get(experiment, experiment))

ax.axhline(0, linewidth=1)
ax.set_title(f"Aglomerado mean residuals ordered by {REFERENCE_EXPERIMENT}")
ax.set_xlabel("Aglomerados ordered by core residual mean")
ax.set_ylabel("Mean residual: y_true - y_pred")
ax.set_xticks(x)
ax.set_xticklabels(ref_order, rotation=90, fontsize=7)
ax.grid(axis="y", alpha=0.3)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "F3_ols_geo_aglo_mean_residual_ranking.png", dpi=160)
plt.show()


## 09. Figure 4 — group mean income vs residual effect

In [ ]:

plot_df = df_aglo.query("experiment in ['ols_core', 'ols_core_aglo_fe', 'ols_core_aglo_plus_time_fe']").copy()

fig, ax = plt.subplots(figsize=(7, 6))

for experiment in ["ols_core", "ols_core_aglo_fe", "ols_core_aglo_plus_time_fe"]:
    tmp = plot_df.query("experiment == @experiment")
    if tmp.empty:
        continue
    ax.scatter(
        tmp["mean_y_true"],
        tmp["mean_residual"],
        s=np.clip(tmp["n"] / 8, 10, 120),
        alpha=0.65,
        label=DISPLAY_LABELS.get(experiment, experiment),
    )

ax.axhline(0, linewidth=1)
ax.set_title("Aglomerado mean income vs residual effect")
ax.set_xlabel("Mean observed log income by aglomerado")
ax.set_ylabel("Mean residual by aglomerado")
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "F4_ols_geo_mean_income_vs_residual.png", dpi=160)
plt.show()


## 10. FE stability across geographic settings

In [ ]:

fe_compare_experiments = ["ols_core", "ols_core_aglo_fe", "ols_core_aglo_plus_time_fe"]

df_fe_wide = (
    df_aglo
    .query("experiment in @fe_compare_experiments")
    .pivot_table(
        index="group_value",
        columns="experiment",
        values="mean_residual",
        aggfunc="first",
    )
)

# preserve reference order where possible
df_fe_wide = df_fe_wide.reindex([g for g in ref_order if g in df_fe_wide.index])

fe_corr = df_fe_wide.corr()
fe_corr.to_csv(TABLE_DIR / "T6_ols_geo_aglo_residual_correlation.csv")

display(fe_corr.round(4))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(fe_corr, vmin=-1, vmax=1)
ax.set_xticks(range(len(fe_corr.columns)))
ax.set_yticks(range(len(fe_corr.index)))
ax.set_xticklabels([DISPLAY_LABELS.get(c, c) for c in fe_corr.columns], rotation=35, ha="right")
ax.set_yticklabels([DISPLAY_LABELS.get(c, c) for c in fe_corr.index])
ax.set_title("Correlation of aglomerado residual effects")
fig.colorbar(im, ax=ax, label="correlation")
fig.tight_layout()
fig.savefig(FIG_DIR / "F5_ols_geo_aglo_effect_correlation_heatmap.png", dpi=160)
plt.show()

if {"ols_core_aglo_fe", "ols_core_aglo_plus_time_fe"}.issubset(df_fe_wide.columns):
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(df_fe_wide["ols_core_aglo_fe"], df_fe_wide["ols_core_aglo_plus_time_fe"], alpha=0.75)
    lim = np.nanmax(np.abs(df_fe_wide[["ols_core_aglo_fe", "ols_core_aglo_plus_time_fe"]].to_numpy()))
    ax.plot([-lim, lim], [-lim, lim], linestyle="--", linewidth=1)
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    ax.set_title("Aglomerado residual effects: aglo FE vs aglo+time FE")
    ax.set_xlabel("Aglomerado FE setting")
    ax.set_ylabel("Aglomerado + time FE setting")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "F6_ols_geo_aglo_fe_vs_aglo_plus_time_fe.png", dpi=160)
    plt.show()


## 11. Top agglomerate contributors

In [ ]:

df_contrib = df_aglo.copy()

df_contrib["weight"] = df_contrib["n"] / df_contrib.groupby("experiment")["n"].transform("sum")
df_contrib["variance_contribution"] = df_contrib["weight"] * (df_contrib["mean_residual"] ** 2)

top_contributors = (
    df_contrib
    .sort_values(["experiment", "variance_contribution"], ascending=[True, False])
    .groupby("experiment")
    .head(12)
    [[
        "experiment",
        "group_value",
        "n",
        "mean_y_true",
        "mean_y_pred",
        "mean_residual",
        "abs_mean_residual",
        "weight",
        "variance_contribution",
    ]]
    .reset_index(drop=True)
)

top_contributors.to_csv(TABLE_DIR / "T7_ols_geo_top_aglo_contributors.csv", index=False)
top_contributors.round(5)


## 12. Automatic diagnostic summary

In [ ]:

def fmt(x, digits=5):
    if pd.isna(x):
        return "NA"
    return f"{x:.{digits}f}"

notes = []
summary = validation_summary.set_index("experiment")

for exp in ["ols_core_region_fe", "ols_core_aglo_fe", "ols_core_aglo_plus_time_fe"]:
    if exp in summary.index and "ols_core" in summary.index:
        notes.append(
            f"{exp}: ΔR² vs core = {fmt(summary.loc[exp, 'delta_r2_vs_core'])}; "
            f"ΔMAE = {fmt(summary.loc[exp, 'delta_mae_vs_core'])}; "
            f"ΔRMSE = {fmt(summary.loc[exp, 'delta_rmse_vs_core'])}."
        )

v = geo_variance_decomposition.query("split == 'validation'")
for exp in ["ols_core", "ols_core_region_fe", "ols_core_aglo_fe", "ols_core_aglo_plus_time_fe"]:
    for group_type in ["Region", "AGLOMERADO"]:
        row = v.query("experiment == @exp and group_type == @group_type")
        if not row.empty:
            row = row.iloc[0]
            notes.append(
                f"{exp} / {group_type}: sd(group mean residual) = {fmt(row['sd_group_mean_residual'])}; "
                f"share Var(y) = {fmt(row['share_var_y_pct'])}%."
            )

if {"ols_core_aglo_fe", "ols_core_aglo_plus_time_fe"}.issubset(fe_corr.columns):
    corr_value = fe_corr.loc["ols_core_aglo_fe", "ols_core_aglo_plus_time_fe"]
    notes.append(
        f"Aglomerado residual-effect stability between aglo FE and aglo+time FE: correlation = {fmt(corr_value)}."
    )

DIAGNOSTIC_SUMMARY = "\n".join(f"- {note}" for note in notes)
print(DIAGNOSTIC_SUMMARY)

(TABLE_DIR / "T8_ols_geo_diagnostic_notes.txt").write_text(DIAGNOSTIC_SUMMARY, encoding="utf-8")



## 13. Outputs written

Tables:

```text
reports/notebook_outputs/ols_geographic_effects/tables/
```

Figures:

```text
reports/notebook_outputs/ols_geographic_effects/figures/
```

Key thesis candidates:

```text
T2_ols_geo_validation_vs_core.csv
T5_ols_geo_variance_decomposition.csv
T7_ols_geo_top_aglo_contributors.csv
F1_ols_geo_delta_r2_vs_core.png
F2_ols_geo_variance_share_region_vs_aglo.png
F3_ols_geo_aglo_mean_residual_ranking.png
F4_ols_geo_mean_income_vs_residual.png
F6_ols_geo_aglo_fe_vs_aglo_plus_time_fe.png
```
